# Ethiopia Food Prices — 13 FINAL MODELS

Finalize the recommended production models, generate the final forecast, and package the model artifacts and operating thresholds for reporting and deployment.


In [4]:
import json
import pandas as pd
from pathlib import Path
import joblib

root = Path('../')
clean_df = pd.read_csv(root / 'data' / 'eth_food_prices_clean.csv', parse_dates=['date'])
clean_df = clean_df.sort_values(['market', 'commodity', 'date']).reset_index(drop=True)
print('Data loaded:', clean_df.shape)


Data loaded: (62875, 21)


In [5]:
focus_market = 'Addis Ababa'
focus_commodity = 'Maize (white)'
focus_unit = '100 KG'
focus_pricetype = 'Wholesale'

series = clean_df[
    (clean_df['market'] == focus_market)
    & (clean_df['commodity'] == focus_commodity)
    & (clean_df['pricetype'] == focus_pricetype)
    & (clean_df['unit'] == focus_unit)
].copy()
series = series.sort_values('date').reset_index(drop=True)
series['date'] = pd.to_datetime(series['date'])
series['year'] = series['date'].dt.year
series['month_of_year'] = series['date'].dt.month
series['lag_1'] = series['usdprice'].shift(1)
series['lag_3'] = series['usdprice'].shift(3)
series['lag_6'] = series['usdprice'].shift(6)

for window in [3, 6, 12]:
    series[f'rolling_mean_{window}'] = series['usdprice'].shift(1).rolling(window=window, min_periods=1).mean()
    series[f'rolling_std_{window}'] = series['usdprice'].shift(1).rolling(window=window, min_periods=2).std()

series['pct_change_1m'] = series['usdprice'].pct_change().shift(1) * 100
series['price_to_mean_ratio'] = series['usdprice'] / series['usdprice'].mean()
series = series.dropna(subset=['lag_1', 'lag_3', 'lag_6']).reset_index(drop=True)

feature_cols = [
    'year', 'month_of_year', 'lag_1', 'lag_3', 'lag_6',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
    'rolling_std_3', 'rolling_std_6', 'rolling_std_12',
    'pct_change_1m', 'price_to_mean_ratio'
]

if series.empty:
    raise ValueError(f'No valid rows found for {focus_market} / {focus_commodity} / {focus_pricetype} / {focus_unit}')

last_row = series.iloc[-1:][feature_cols]
forecast_model = joblib.load(root / 'models' / 'maize_forecast_model.joblib')
next_month_prediction = float(forecast_model.predict(last_row)[0])
print(f'One-step forecast for {focus_commodity} in {focus_market}: {next_month_prediction:.2f}')

# In a production setting, we also keep the classification decision threshold with the model artifact.
threshold_path = root / 'models' / 'spike_threshold.json'
if threshold_path.exists():
    with open(threshold_path, 'r') as f:
        threshold_config = json.load(f)
    threshold_value = threshold_config.get('best_threshold', threshold_config.get('threshold', 0.50))
    print(f'Loaded spike decision threshold: {threshold_value:.2f}')
else:
    print('No saved spike threshold file found; use the default 0.50 cutoff only when deploying.')


One-step forecast for Maize (white) in Addis Ababa: 50.15
Loaded spike decision threshold: 0.33


In [6]:
# Save final model summary and forecast artifact
summary = pd.DataFrame({
    'model_type': ['forecasting', 'spike_classification'],
    'artifact': ['maize_forecast_model.joblib', 'spike_classifier_model.joblib'],
    'decision_threshold': [None, 0.50],
    'status': ['trained', 'trained']
})

threshold_path = root / 'models' / 'spike_threshold.json'
if threshold_path.exists():
    with open(threshold_path, 'r') as f:
        threshold_config = json.load(f)
    threshold_value = threshold_config.get('best_threshold', threshold_config.get('threshold', 0.50))
    summary.loc[summary['model_type'] == 'spike_classification', 'decision_threshold'] = threshold_value

summary.to_csv(root / 'models' / 'final_model_summary.csv', index=False)
print(summary.to_string(index=False))


          model_type                      artifact  decision_threshold  status
         forecasting   maize_forecast_model.joblib                 NaN trained
spike_classification spike_classifier_model.joblib                0.33 trained


This final notebook packages the selected production models, keeps the tuned spike threshold with the classifier artifact, and makes the entire workflow reproducible for reporting, monitoring, and deployment.
